# Escape Heatmaps From Max-Rayleigh Tuned IDs

This notebook creates **one spatial firing heatmap per escape bout** for each tuned cell from:
- `tuned_ids_A_preflip_AND_maxrayleighA.csv`
- `tuned_ids_B_postflip_AND_maxrayleighB.csv`

It resolves session-name mismatches, loads session data, detects escape bouts within each target condition,
and saves one PNG per escape bout.


In [1]:
from pathlib import Path
import re
import gc

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import polars as pl
from IPython.display import Image, display

from behave_analysis.analyze.filtering_data.filtering_functions import filter_video_dataframe


In [2]:
# -----------------------
# Paths / settings
# -----------------------
SAVE_ROOT = Path(r"/ceph/branco/Jasmine_Laurence/rayleigh_analysis/Top2_TunED")
EXPERIMENTAL_DATA_ROOT = Path(r"/ceph/branco/Jasmine_Laurence/Experimental_Data")
OUT_DIR = SAVE_ROOT / "escape_heatmaps_maxrayleigh_tuned_ids"

A_IDS_CSV = SAVE_ROOT / "tuned_ids_A_preflip_AND_maxrayleighA.csv"
B_IDS_CSV = SAVE_ROOT / "tuned_ids_B_postflip_AND_maxrayleighB.csv"

FPS = 40
NBINS = 30
COLORMAP = "magma"
DPI = 250

# Stability controls
SESSION_PREFIX_FILTER = ["JAL4"]   # e.g. ["JAL4"] to start small, then set to None
MAX_CELLS = 100                    # set None for all
MAX_ESCAPES_PER_CELL = 30          # set None for all escapes per cell
MAX_IMAGE_PREVIEWS = 8             # notebook previews from saved PNGs


In [3]:
# -----------------------
# Helpers
# -----------------------
MONTHS = {
    "jan": 1, "january": 1,
    "feb": 2, "february": 2,
    "mar": 3, "march": 3,
    "apr": 4, "april": 4,
    "may": 5,
    "jun": 6, "june": 6,
    "jul": 7, "july": 7,
    "aug": 8, "august": 8,
    "sep": 9, "sept": 9, "september": 9,
    "oct": 10, "october": 10,
    "nov": 11, "november": 11,
    "dec": 12, "december": 12,
}

def _short_month(mm: int) -> str:
    arr = ["jan", "feb", "mar", "apr", "may", "jun", "jul", "aug", "sep", "oct", "nov", "dec"]
    return arr[mm - 1]


def _mouse_num(name: str):
    m = re.search(r"JAL0*(\d+)", name, flags=re.IGNORECASE)
    if m:
        return int(m.group(1))
    m = re.match(r"0*(\d+)_", name)
    if m:
        return int(m.group(1))
    return None


def _flip_num(name: str):
    m = re.search(r"flip[_ ]?(\d+)", name, flags=re.IGNORECASE)
    return int(m.group(1)) if m else None


def _day_month_from_target(name: str):
    m = re.search(r"(\d{1,2})(?:st|nd|rd|th)?([A-Za-z]{3,9})", name, flags=re.IGNORECASE)
    if not m:
        return None
    dd = int(m.group(1))
    mon_raw = m.group(2).lower()
    if mon_raw not in MONTHS:
        return None
    mm = MONTHS[mon_raw]
    return (dd, mm)


def _day_month_from_discovered(name: str):
    m = re.search(r"(20\d{2})_(\d{2})_(\d{2})", name)
    if m:
        mm = int(m.group(2))
        dd = int(m.group(3))
        return (dd, mm)
    return _day_month_from_target(name)


def _target_alias_keys(name: str):
    mouse = _mouse_num(name)
    flip = _flip_num(name)
    dm = _day_month_from_target(name)
    keys = []
    if mouse is None:
        return keys
    if flip is not None and dm is not None:
        keys.append(f"m{mouse}|f{flip}|d{dm[0]}{_short_month(dm[1])}")
    if dm is not None:
        keys.append(f"m{mouse}|d{dm[0]}{_short_month(dm[1])}")
    if flip is not None:
        keys.append(f"m{mouse}|f{flip}")
    keys.append(f"m{mouse}|raw:{name.lower()}")
    return keys


def _discovered_alias_keys(name: str):
    mouse = _mouse_num(name)
    flip = _flip_num(name)
    dm = _day_month_from_discovered(name)
    keys = []
    if mouse is None:
        return keys
    if flip is not None and dm is not None:
        keys.append(f"m{mouse}|f{flip}|d{dm[0]}{_short_month(dm[1])}")
    if dm is not None:
        keys.append(f"m{mouse}|d{dm[0]}{_short_month(dm[1])}")
    if flip is not None:
        keys.append(f"m{mouse}|f{flip}")
    keys.append(f"m{mouse}|raw:{name.lower()}")
    return keys


def resolve_session_names(target_sessions, discovered_session_names):
    key_to_discovered = {}
    for ds in discovered_session_names:
        for k in _discovered_alias_keys(ds):
            key_to_discovered.setdefault(k, []).append(ds)

    mapping, unresolved, ambiguous = {}, [], {}
    for ts in sorted(set(target_sessions)):
        candidates = []
        for k in _target_alias_keys(ts):
            hits = key_to_discovered.get(k, [])
            if len(hits) == 1:
                mapping[ts] = hits[0]
                candidates = []
                break
            if len(hits) > 1:
                candidates = hits
        else:
            if candidates:
                ambiguous[ts] = sorted(set(candidates))
            else:
                unresolved.append(ts)
    return mapping, unresolved, ambiguous


def load_targets() -> pd.DataFrame:
    for p in [A_IDS_CSV, B_IDS_CSV]:
        if not p.exists():
            raise FileNotFoundError(f"Missing IDs CSV: {p}")

    a_ids = pd.read_csv(A_IDS_CSV)[["session", "condition", "cluster_id"]].copy()
    b_ids = pd.read_csv(B_IDS_CSV)[["session", "condition", "cluster_id"]].copy()
    a_ids["tuned_label"] = "A_only"
    b_ids["tuned_label"] = "B_only"

    out = pd.concat([a_ids, b_ids], ignore_index=True)
    out["cluster_id"] = pd.to_numeric(out["cluster_id"], errors="coerce")
    out = out.dropna(subset=["session", "cluster_id"]).copy()
    out["cluster_id"] = out["cluster_id"].astype(int)
    return out.drop_duplicates().sort_values(["session", "cluster_id", "condition"]).reset_index(drop=True)


def find_session_processed_dirs(root: Path) -> dict:
    if not root.exists():
        raise FileNotFoundError(f"Experimental data root not found: {root}")

    session_map = {}
    for p in root.rglob("full_video_dataframe.csv"):
        if p.parent.name != "processed_data":
            continue
        session_map.setdefault(p.parent.parent.name, p.parent)
    return session_map


def ensure_bool_columns(df: pl.DataFrame) -> pl.DataFrame:
    out = df
    for col in ["shelter", "barrier_present", "barrier_flipped", "EscapePeriod", "OutofshelterIdx", "homingPeriod"]:
        if col in out.columns:
            out = out.with_columns(pl.col(col).cast(pl.Boolean))
    return out


def add_bin_edges(vpos_pd: pd.DataFrame, nbins: int):
    x_min, x_max = np.nanmin(vpos_pd["x"].values), np.nanmax(vpos_pd["x"].values)
    y_min, y_max = np.nanmin(vpos_pd["y"].values), np.nanmax(vpos_pd["y"].values)
    eps = 1e-9
    x_edges = np.linspace(x_min, x_max + eps, nbins + 1)
    y_edges = np.linspace(y_min, y_max + eps, nbins + 1)
    return x_edges, y_edges


def apply_bins(df_pd: pd.DataFrame, x_edges: np.ndarray, y_edges: np.ndarray):
    nbins_x = len(x_edges) - 1
    nbins_y = len(y_edges) - 1
    df_pd["x_bins"] = np.clip(np.digitize(df_pd["x"].values, x_edges) - 1, 0, nbins_x - 1)
    df_pd["y_bins"] = np.clip(np.digitize(df_pd["y"].values, y_edges) - 1, 0, nbins_y - 1)


def load_session_data(processed_dir: Path):
    video_csv = processed_dir / "full_video_dataframe.csv"
    spike_csv = processed_dir / "spike_count_by_frame_and_goodcluster.csv"
    if not video_csv.exists() or not spike_csv.exists():
        return None, None

    vdf_raw = ensure_bool_columns(pl.read_csv(video_csv))
    sdf_raw = pl.read_csv(spike_csv)
    sdf = sdf_raw.rename({"spike_aligned_to_frame": "frame"}).with_columns(pl.col("frame").cast(pl.Int64))
    sdf_pd = sdf.select(["frame", "spike_clusters", "spike_count"]).to_pandas()
    return vdf_raw, sdf_pd


def extract_escape_bouts(vdf_raw: pl.DataFrame, condition: str):
    base = filter_video_dataframe(vdf_raw, condition, outofshelter=True, exclude_escape=False)
    if "EscapePeriod" not in base.columns:
        return []

    esc = base.filter(pl.col("EscapePeriod") == True)
    if esc.is_empty():
        return []

    frames = np.sort(esc["frames"].to_numpy().astype(int))
    if frames.size == 0:
        return []

    splits = np.where(np.diff(frames) > 1)[0] + 1
    chunks = np.split(frames, splits)
    bouts = [c for c in chunks if c.size > 0]
    return bouts


def robust_min_max(arr: np.ndarray, low=2, high=98):
    vals = arr[np.isfinite(arr)]
    if vals.size == 0:
        return 0.0, 1.0
    vmin = np.percentile(vals, low)
    vmax = np.percentile(vals, high)
    if np.isclose(vmin, vmax):
        vmax = vmin + 1e-6
    return float(vmin), float(vmax)


In [ ]:
# -----------------------
# Load targets and resolve sessions
# -----------------------
targets = load_targets()
if SESSION_PREFIX_FILTER:
    pref = tuple(str(x).lower() for x in SESSION_PREFIX_FILTER)
    targets = targets[targets["session"].astype(str).str.lower().str.startswith(pref)].copy()

if MAX_CELLS is not None:
    targets = targets.head(int(MAX_CELLS)).copy()

session_to_processed = find_session_processed_dirs(EXPERIMENTAL_DATA_ROOT)
resolved_map, unresolved, ambiguous = resolve_session_names(
    target_sessions=targets["session"].tolist(),
    discovered_session_names=list(session_to_processed.keys()),
)

targets["session_discovered"] = targets["session"].map(resolved_map)
keep = targets.dropna(subset=["session_discovered"]).copy()

print(f"Targets after filters: {len(targets):,}")
print(f"Resolved rows: {len(keep):,}")
print(f"Resolved sessions: {keep['session'].nunique():,}")
print(f"Unresolved sessions: {len(unresolved):,}")
print(f"Ambiguous sessions: {len(ambiguous):,}")
if unresolved:
    print("First unresolved:", unresolved[:15])

keep.head()


Targets after filters: 20
Resolved rows: 20
Resolved sessions: 4
Unresolved sessions: 0
Ambiguous sessions: 0


,session,condition,cluster_id,tuned_label,session_discovered
1,JAL4_11thSept,barrier_post_flip,2,B_only,004_flip_puff2_2023_09_11T09_32_25
2,JAL4_11thSept,barrier_pre_flip,63,A_only,004_flip_puff2_2023_09_11T09_32_25
3,JAL4_11thSept,barrier_pre_flip,78,A_only,004_flip_puff2_2023_09_11T09_32_25
4,JAL4_11thSept,barrier_pre_flip,146,A_only,004_flip_puff2_2023_09_11T09_32_25
5,JAL4_11thSept,barrier_post_flip,486,B_only,004_flip_puff2_2023_09_11T09_32_25


: 

In [ ]:
# -----------------------
# Build per-escape heatmaps and save
# -----------------------
OUT_DIR.mkdir(parents=True, exist_ok=True)

saved = 0
shown = 0
skipped_sessions = 0
skipped_cells = 0
cells_no_escape = 0

for session, grp in keep.groupby("session", sort=True):
    sess_key = str(grp["session_discovered"].iloc[0])
    processed_dir = session_to_processed.get(sess_key)
    if processed_dir is None:
        skipped_sessions += len(grp)
        print(f"[missing session] {session} -> {sess_key}")
        continue

    vdf_raw, sdf_pd = load_session_data(processed_dir)
    if vdf_raw is None or sdf_pd is None:
        skipped_sessions += len(grp)
        print(f"[missing csvs] {session} -> {processed_dir}")
        continue

    spike_clusters = set(pd.to_numeric(sdf_pd["spike_clusters"], errors="coerce").dropna().astype(int).unique())

    vpos = vdf_raw.select(["frames", "mouse_x_position", "mouse_y_position"]).rename(
        {"frames": "frame", "mouse_x_position": "x", "mouse_y_position": "y"}
    ).to_pandas()
    x_edges, y_edges = add_bin_edges(vpos, NBINS)

    print(f"\nSession {session} ({sess_key}) | target rows={len(grp)}")

    for _, row in grp.iterrows():
        cluster_id = int(row["cluster_id"])
        tuned_label = str(row["tuned_label"])
        condition = str(row["condition"])

        if cluster_id not in spike_clusters:
            skipped_cells += 1
            print(f"  [missing cluster] unit {cluster_id}")
            continue

        bouts = extract_escape_bouts(vdf_raw, condition)
        if MAX_ESCAPES_PER_CELL is not None:
            bouts = bouts[: int(MAX_ESCAPES_PER_CELL)]
        if len(bouts) == 0:
            cells_no_escape += 1
            print(f"  [no escapes] unit {cluster_id} | cond={condition}")
            continue

        clu_spk = sdf_pd[sdf_pd["spike_clusters"] == cluster_id][["frame", "spike_count"]].copy()
        if clu_spk.empty:
            skipped_cells += 1
            print(f"  [no spikes] unit {cluster_id}")
            continue

        for i, bout_frames in enumerate(bouts, start=1):
            esc_df = (
                vdf_raw.filter(pl.col("frames").is_in(bout_frames.tolist()))
                .select(["frames", "mouse_x_position", "mouse_y_position"])
                .rename({"frames": "frame", "mouse_x_position": "x", "mouse_y_position": "y"})
                .with_columns(pl.col("frame").cast(pl.Int64))
                .to_pandas()
            )

            if esc_df.empty:
                continue

            apply_bins(esc_df, x_edges, y_edges)
            occ = esc_df.groupby(["y_bins", "x_bins"]).size().unstack(fill_value=0)
            occ = occ.reindex(index=np.arange(NBINS), columns=np.arange(NBINS), fill_value=0)

            df_unit = esc_df.merge(clu_spk, on="frame", how="left")
            df_unit["spike_count"] = df_unit["spike_count"].fillna(0)
            spk_df = df_unit[df_unit["spike_count"] > 0]

            if spk_df.empty:
                spk = pd.DataFrame(0, index=occ.index, columns=occ.columns)
            else:
                spk = spk_df.groupby(["y_bins", "x_bins"])["spike_count"].sum().unstack(fill_value=0)
                spk = spk.reindex(index=occ.index, columns=occ.columns, fill_value=0)

            with np.errstate(divide="ignore", invalid="ignore"):
                rate = spk.values / np.where(occ.values == 0, np.nan, occ.values) * FPS

            rate_masked = np.ma.array(rate, mask=~np.isfinite(rate))
            vmin, vmax = robust_min_max(rate, 2, 98)

            cmap = plt.get_cmap(COLORMAP).copy()
            cmap.set_bad(color="0.8")

            fig, ax = plt.subplots(1, 1, figsize=(4.8, 4.8))
            im = ax.imshow(
                rate_masked,
                origin="lower",
                cmap=cmap,
                vmin=vmin,
                vmax=vmax,
                interpolation="nearest",
                extent=[x_edges[0], x_edges[-1], y_edges[0], y_edges[-1]],
                aspect="equal",
            )
            ax.set_xticks([])
            ax.set_yticks([])
            for s in ["top", "right", "left", "bottom"]:
                ax.spines[s].set_visible(False)
            ax.invert_yaxis()

            total_spk = int(df_unit["spike_count"].sum())
            n_frames = int(len(df_unit))
            ax.set_title(f"escape {i} | spikes={total_spk} | frames={n_frames}", fontsize=10)

            cbar = fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
            cbar.set_label("Spikes/s", rotation=90)
            fig.suptitle(f"{session} | unit {cluster_id} | {tuned_label} | {condition}", fontsize=11)
            fig.subplots_adjust(left=0.05, right=0.90, top=0.87, bottom=0.05)

            out_name = f"{session}__unit_{cluster_id}__{tuned_label}__{condition}__escape{i:02d}.png"
            out_path = OUT_DIR / out_name
            fig.savefig(out_path, dpi=DPI, bbox_inches="tight")
            plt.close(fig)

            saved += 1
            print(f"  saved: {out_name}")

            if shown < MAX_IMAGE_PREVIEWS:
                display(Image(filename=str(out_path)))
                shown += 1

            del esc_df, occ, df_unit, spk_df, spk, rate, rate_masked
            gc.collect()

        del clu_spk
        gc.collect()

    del vdf_raw, sdf_pd, vpos
    gc.collect()

print("\nDone")
print(f"Saved heatmaps: {saved}")
print(f"Preview images shown: {shown}")
print(f"Skipped rows (session/csv issues): {skipped_sessions}")
print(f"Skipped cells (cluster/spike issues): {skipped_cells}")
print(f"Cells with no escapes in target condition: {cells_no_escape}")
print(f"Output dir: {OUT_DIR}")
